In [10]:
import pandas as pd
from pathlib import Path
import numpy as np

COMM_AREA_AGG_PATH = Path("../data/processed/community_areas_agg_df.parquet")
comm_area_agg_df = pd.read_parquet(COMM_AREA_AGG_PATH)

In [11]:
comm_area_agg_df.head()

,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX,UniqueNetworkManagersCount,MeanSafetyScore,MeanFamilyInvolvementScore,MeanEnvironmentScore,MeanInstructionScore,MeanStudentAttendance,TotalCrimes,UniqueCrimeCategories,ArrestsCount
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0,2,43.000000,36.400000,43.250000,36.750000,93.050000,6.0,6.0,3.0
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0,2,67.222222,59.571429,58.444444,56.555556,94.922222,7.0,5.0,2.0
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0,2,52.000000,48.500000,59.000000,58.000000,94.542857,4.0,1.0,4.0
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0,2,67.200000,55.000000,46.600000,44.600000,92.840000,3.0,3.0,1.0
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0,2,85.166667,83.666667,65.500000,51.166667,95.542857,4.0,4.0,2.0


In [12]:
# school quality score
comm_area_agg_df["SchoolQualityScore"] = comm_area_agg_df[
        [
            "MeanSafetyScore",
            "MeanFamilyInvolvementScore",
            "MeanEnvironmentScore",
            "MeanInstructionScore"
        ]
    ].mean(axis=1)

In [13]:
# crimes per income
comm_area_agg_df["CrimesPerIncome"] = comm_area_agg_df.TotalCrimes / comm_area_agg_df.PER_CAPITA_INCOME

In [14]:
# crimes per crime category
comm_area_agg_df["CrimesPerCrimeCategory"] = np.where(comm_area_agg_df.UniqueCrimeCategories > 0, 
                                                      comm_area_agg_df.TotalCrimes / comm_area_agg_df.UniqueCrimeCategories, 
                                                      0)

In [15]:
# arrests per crime
# use np.where() to prevent NaN when values are 0
comm_area_agg_df["ArrestsPerCrime"] = np.where(comm_area_agg_df.TotalCrimes > 0,
    comm_area_agg_df.ArrestsCount / comm_area_agg_df.TotalCrimes,
    0)

In [16]:
# check final result
comm_area_agg_df.head()

,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX,UniqueNetworkManagersCount,...,MeanEnvironmentScore,MeanInstructionScore,MeanStudentAttendance,TotalCrimes,UniqueCrimeCategories,ArrestsCount,SchoolQualityScore,CrimesPerIncome,CrimesPerCrimeCategory,ArrestsPerCrime
0,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0,2,...,43.250000,36.750000,93.050000,6.0,6.0,3.0,39.850000,0.000251,1.0,0.500000
1,2.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0,2,...,58.444444,56.555556,94.922222,7.0,5.0,2.0,60.448413,0.000304,1.4,0.285714
2,3.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0,2,...,59.000000,58.000000,94.542857,4.0,1.0,4.0,54.375000,0.000112,4.0,1.000000
3,4.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0,2,...,46.600000,44.600000,92.840000,3.0,3.0,1.0,53.350000,0.000080,1.0,0.333333
4,5.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0,2,...,65.500000,51.166667,95.542857,4.0,4.0,2.0,71.375000,0.000070,1.0,0.500000


In [17]:
# save it
comm_area_agg_df.to_parquet("../data/processed/community_areas_agg_fe_df.parquet", index=False)